In [1]:
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from scipy.ndimage import label, binary_opening, binary_closing, binary_fill_holes

In [2]:
project_root = Path("..").resolve()

train_dir = project_root / "data" / "train"
output_dir = project_root / "outputs"
output_dir.mkdir(exist_ok=True)

subject = "subj001"
subject_dir = train_dir / subject

image_path = subject_dir / f"{subject}_image.nii.gz"
artifact_path = subject_dir / f"{subject}_artifact.nii.gz"

print("Image exists:", image_path.exists())
print("Artifact exists:", artifact_path.exists())

Image exists: True
Artifact exists: True


In [3]:
img = nib.load(image_path)
I = img.get_fdata()

artifact_img = nib.load(artifact_path)
A = artifact_img.get_fdata()

print("MRI shape:", I.shape)
print("Artifact shape:", A.shape)
print("Artifact values:", np.unique(A))

MRI shape: (192, 224, 192)
Artifact shape: (192, 224, 192)
Artifact values: [0. 1.]


In [4]:
def normalize_volume(I):
    """
    Normalize MRI volume using mean/std.
    """
    return (I - np.mean(I)) / (np.std(I) + 1e-8)


def dice_score(pred, truth):
    """
    Compute Dice similarity between predicted mask and ground truth mask.
    """
    pred = pred.astype(bool)
    truth = truth.astype(bool)

    intersection = np.logical_and(pred, truth).sum()
    denom = pred.sum() + truth.sum()

    if denom == 0:
        return 1.0

    return 2 * intersection / denom


def keep_largest_components(mask, min_size=100, max_components=3):
    """
    Keep the largest connected components in a binary mask.
    """
    labeled, num = label(mask)

    if num == 0:
        return np.zeros_like(mask, dtype=bool)

    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0

    component_ids = np.argsort(sizes)[::-1]

    out = np.zeros_like(mask, dtype=bool)
    kept = 0

    for comp_id in component_ids:
        if comp_id == 0:
            continue
        if sizes[comp_id] < min_size:
            continue

        out |= (labeled == comp_id)
        kept += 1

        if kept >= max_components:
            break

    return out

In [21]:
def predict_artifact_mask(I, hemisphere="right", debug=False):
    """
    Automatic classical valve artifact detector.
    Uses MRI + hemisphere metadata.
    Does NOT use ground-truth artifact mask.
    """

    I_norm = normalize_volume(I)

    # -----------------------------
    # 1. Rough head mask
    # -----------------------------
    head_thresh = np.percentile(I_norm, 30)
    head_mask = I_norm > head_thresh

    head_mask = binary_closing(head_mask, iterations=2)
    head_mask = binary_fill_holes(head_mask)

    # -----------------------------
    # 2. Dark voxels inside head
    # -----------------------------
    dark_thresh = np.percentile(I_norm[head_mask], 15)
    dark_mask = I_norm < dark_thresh

    # -----------------------------
    # 3. Spatial search box
    # -----------------------------
    x = np.arange(I.shape[0])[:, None, None]
    y = np.arange(I.shape[1])[None, :, None]
    z = np.arange(I.shape[2])[None, None, :]

    # Use hemisphere metadata
    if hemisphere == "right":
        side_mask = (x > int(0.55 * I.shape[0])) & (x < int(0.95 * I.shape[0]))
    else:
        side_mask = (x > int(0.05 * I.shape[0])) & (x < int(0.45 * I.shape[0]))

    # Avoid extreme front/back edges and top/bottom junk
    y_mask = (y > int(0.35 * I.shape[1])) & (y < int(0.80 * I.shape[1]))
    z_mask = (z > int(0.35 * I.shape[2])) & (z < int(0.70 * I.shape[2]))

    search_region = side_mask & y_mask & z_mask

    # -----------------------------
    # 4. Candidate mask
    # -----------------------------
    candidate = head_mask & dark_mask & search_region

    candidate = binary_opening(candidate, iterations=1)
    candidate = binary_closing(candidate, iterations=1)

    # -----------------------------
    # 5. Connected components
    # -----------------------------
    labeled, num = label(candidate)

    if num == 0:
        if debug:
            print("No candidate components found.")
        return np.zeros_like(I, dtype=bool)

    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0

    best_score = -np.inf
    best_component = None

    for comp_id in range(1, num + 1):
        comp = labeled == comp_id
        size = sizes[comp_id]

        # Filter unreasonable sizes
        if size < 50 or size > 25000:
            continue

        coords = np.argwhere(comp)
        x_min, y_min, z_min = coords.min(axis=0)
        x_max, y_max, z_max = coords.max(axis=0)

        dx = x_max - x_min + 1
        dy = y_max - y_min + 1
        dz = z_max - z_min + 1

        # Avoid giant sheets/long structures
        if dx > 90 or dy > 90 or dz > 70:
            continue

        bbox_volume = dx * dy * dz
        fill_ratio = size / bbox_volume
        mean_intensity = I_norm[comp].mean()

        # Score: prefer dark, compact, reasonable-size components
        score = (-mean_intensity) + 2.0 * fill_ratio + 0.00005 * size

        if score > best_score:
            best_score = score
            best_component = comp

    if best_component is None:
        if debug:
            print("No component passed filters.")
        return np.zeros_like(I, dtype=bool)

    # -----------------------------
    # 6. Crop around best component
    # -----------------------------
    coords = np.argwhere(best_component)
    x_min, y_min, z_min = coords.min(axis=0)
    x_max, y_max, z_max = coords.max(axis=0)

    pad = 20

    x1 = max(x_min - pad, 0)
    x2 = min(x_max + pad, I.shape[0])

    y1 = max(y_min - pad, 0)
    y2 = min(y_max + pad, I.shape[1])

    z1 = max(z_min - pad, 0)
    z2 = min(z_max + pad, I.shape[2])

    crop = I_norm[x1:x2, y1:y2, z1:z2]

    # -----------------------------
    # 7. Local dark threshold in crop
    # -----------------------------
    local_thresh = np.percentile(crop, 50)
    crop_pred = crop < local_thresh

    crop_pred = binary_opening(crop_pred, iterations=1)
    crop_pred = binary_closing(crop_pred, iterations=2)

    # Keep largest connected component inside crop
    crop_labeled, crop_num = label(crop_pred)

    if crop_num == 0:
        if debug:
            print("No crop components found.")
        return np.zeros_like(I, dtype=bool)

    crop_sizes = np.bincount(crop_labeled.ravel())
    crop_sizes[0] = 0
    largest = np.argmax(crop_sizes)
    crop_pred = crop_labeled == largest

    # -----------------------------
    # 8. Put crop result back into full volume
    # -----------------------------
    pred = np.zeros_like(I, dtype=bool)
    pred[x1:x2, y1:y2, z1:z2] = crop_pred

    if debug:
        print("Hemisphere:", hemisphere)
        print("Dark threshold:", dark_thresh)
        print("Best score:", best_score)
        print("Best component size:", best_component.sum())
        print("Crop box:", (x1, x2, y1, y2, z1, z2))
        print("Local threshold:", local_thresh)
        print("Final voxels:", pred.sum())

    return pred

In [22]:
hemisphere = meta["hemisphere"]

M_pred = predict_artifact_mask(I, hemisphere=hemisphere, debug=True)

print("Pred voxels:", M_pred.sum())
print("GT voxels:", A.sum())
print("Overlap:", np.logical_and(M_pred > 0, A > 0).sum())
print("Dice:", dice_score(M_pred, A))

Hemisphere: right
Dark threshold: -0.7051348065773021
Best score: 1.202636573638349
Best component size: 93
Crop box: (np.int64(155), 192, np.int64(123), np.int64(170), np.int64(95), np.int64(142))
Local threshold: -0.7002884641025794
Final voxels: 27998
Pred voxels: 27998
GT voxels: 11995.0
Overlap: 6123
Dice: 0.3062035856274848


In [11]:
import json

json_path = subject_dir / f"{subject}.json"

with open(json_path, "r") as f:
    meta = json.load(f)

meta


{'subject_id': 'subj001',
 'hemisphere': 'right',
 'catheter_length_mm': 55.28,
 'spacing_zyx_mm': [1.0, 1.0, 1.0]}

In [23]:
import json

results = []

for subject_dir in sorted(train_dir.iterdir())[:10]:
    if not subject_dir.is_dir():
        continue

    subject = subject_dir.name

    image_path = subject_dir / f"{subject}_image.nii.gz"
    artifact_path = subject_dir / f"{subject}_artifact.nii.gz"
    json_path = subject_dir / f"{subject}.json"

    img = nib.load(image_path)
    I = img.get_fdata()

    A = nib.load(artifact_path).get_fdata()

    with open(json_path, "r") as f:
        meta = json.load(f)

    hemisphere = meta["hemisphere"]

    M_pred = predict_artifact_mask(I, hemisphere=hemisphere, debug=False)

    dice = dice_score(M_pred, A)
    overlap = np.logical_and(M_pred > 0, A > 0).sum()

    results.append({
        "subject": subject,
        "hemisphere": hemisphere,
        "dice": dice,
        "pred_voxels": M_pred.sum(),
        "gt_voxels": A.sum(),
        "overlap": overlap
    })

    print(
        f"{subject} | hemi={hemisphere} | "
        f"Dice={dice:.4f} | overlap={overlap} | "
        f"pred={M_pred.sum()} | gt={A.sum()}"
    )

subj001 | hemi=right | Dice=0.3062 | overlap=6123 | pred=27998 | gt=11995.0
subj002 | hemi=right | Dice=0.1743 | overlap=11461 | pred=117468 | gt=14073.0
subj003 | hemi=right | Dice=0.3554 | overlap=15126 | pred=68434 | gt=16692.0
subj004 | hemi=right | Dice=0.1618 | overlap=8489 | pred=94992 | gt=9912.0
subj005 | hemi=right | Dice=0.0000 | overlap=0 | pred=39804 | gt=8523.0
subj006 | hemi=right | Dice=0.0000 | overlap=0 | pred=29757 | gt=10558.0
subj007 | hemi=right | Dice=0.2517 | overlap=4966 | pred=26498 | gt=12960.0
subj008 | hemi=right | Dice=0.0000 | overlap=0 | pred=24462 | gt=13104.0
subj009 | hemi=right | Dice=0.0000 | overlap=0 | pred=22020 | gt=11173.0
subj010 | hemi=right | Dice=0.1515 | overlap=3778 | pred=35465 | gt=14425.0
